In [3]:
import torch
from datasets import load_dataset

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments
)

from peft import LoraConfig
from trl import SFTTrainer

In [4]:
print("CUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA Available: True
GPU: NVIDIA GeForce RTX 5060 Ti


In [5]:
train_dataset = load_dataset(
    "json",
    data_files="../data/qwen_scanner_finetune_train.jsonl",
    split="train"
)

val_dataset = load_dataset(
    "json",
    data_files="../data/qwen_scanner_finetune_val.jsonl",
    split="train"
)

print(train_dataset)
print(val_dataset)
print(train_dataset[0])

Dataset({
    features: ['messages'],
    num_rows: 216
})
Dataset({
    features: ['messages'],
    num_rows: 54
})
{'messages': [{'role': 'user', 'content': 'You are an OSS compliance assistant.\n\nClassify this scanner output.\n\nPackage: example-package\nVersion: 1.0.0\nEcosystem: python\nPackage Manager: pip\nLicense: CC0-1.0\nLicense Family: Public Domain\nDependency Scope: runtime\nDirect or Transitive: direct\nDevelopment Dependency: no\nProject Type: enterprise workflow system\nDistribution Model: distributed\nUsage: backend framework\nLinking Type: dynamic\nNetwork Exposed: yes\nCommercial Use: yes\nAttribution Notice: preserved\nLicense Text: included\nSource Modified: no\nRedistribution: no\nLicense Confidence: high\n\nReturn exactly:\nRisk: Low/Medium/High\nReason: one short sentence'}, {'role': 'assistant', 'content': 'Risk: Low\nReason: License creates minimal compliance burden for commercial and hosted usage'}]}


In [6]:
model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.float16
)

print("Base Qwen loaded successfully.")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Base Qwen loaded successfully.


In [7]:
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

print(peft_config)

LoraConfig(task_type='CAUSAL_LM', peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, peft_version='0.19.1', base_model_name_or_path=None, revision=None, inference_mode=False, r=16, target_modules={'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj', 'k_proj', 'q_proj'}, exclude_modules=None, lora_alpha=32, lora_dropout=0.05, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, lora_ga_config=None, use_dora=False, alora_invocation_tokens=None, use_qalora=False, qalora_group_size=16, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False), lora_bias=False, target_parameters=None, use_bdlora=None, arrow_config=None, ensure_weight_tying=False)


In [8]:
training_args = TrainingArguments(
    output_dir="../models/qwen_oss_compliance_scanner_lora",

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,

    gradient_accumulation_steps=8,

    learning_rate=5e-5,
    num_train_epochs=3,

    logging_steps=5,

    save_strategy="epoch",
    eval_strategy="epoch",

    fp16=True,

    report_to="none"
)

In [9]:
trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    peft_config=peft_config,
    args=training_args
)

print("Trainer created successfully.")

Tokenizing train dataset:   0%|          | 0/216 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/54 [00:00<?, ? examples/s]

Trainer created successfully.


In [10]:
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Epoch,Training Loss,Validation Loss
1,1.047934,0.689949
2,0.299791,0.255874
3,0.201491,0.205183


TrainOutput(global_step=81, training_loss=0.8685161406979148, metrics={'train_runtime': 105.5956, 'train_samples_per_second': 6.137, 'train_steps_per_second': 0.767, 'total_flos': 988886330376192.0, 'train_loss': 0.8685161406979148})

In [11]:
trainer.save_model("../models/qwen_oss_compliance_scanner_lora")

print("Scanner-style Qwen LoRA model saved successfully.")

Scanner-style Qwen LoRA model saved successfully.


In [13]:
from peft import PeftModel

test_scenario = """
Package: numpy
Version: 1.26.0
Ecosystem: python
Package Manager: pip
License: BSD-3-Clause
License Family: Permissive
Dependency Scope: runtime
Direct or Transitive: direct
Development Dependency: no
Project Type: commercial SaaS platform
Distribution Model: hosted
Usage: library
Linking Type: dynamic
Network Exposed: yes
Commercial Use: yes
Attribution Notice: preserved
License Text: included
Source Modified: no
Redistribution: no
License Confidence: high
"""

prompt = f"""
You are an OSS compliance assistant.

Classify this scanner output.

{test_scenario}

Return exactly in this format:
Risk: <Low|Medium|High>
Reason: <one short sentence>

Do not omit "Risk:" or "Reason:".
""".strip()

messages = [
    {"role": "user", "content": prompt}
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer([text], return_tensors="pt").to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=80,
    do_sample=False
)

response = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[-1]:],
    skip_special_tokens=True
)

print(response)

Risk: Low
Reason: Commercial SaaS platforms often require permissive licenses to comply with licensing regulations and legal requirements for software distribution
